# Quantum Support Vector Regression: Methods M1, M2, and M3

This notebook adds three **Quantum Support Vector Regression (QSVR)** approaches to the shallow-water bathymetry project. All methods use the same six angle-encoded spectral features and the same train/test files created by `Encoding.ipynb`.

- **QSVR M1 — Local Fidelity Kernel:** a `ZFeatureMap` without entanglement.
- **QSVR M2 — Entangled Fidelity Kernel:** a `ZZFeatureMap` with linear two-qubit interactions.
- **QSVR M3 — Projected Quantum Kernel:** PennyLane quantum features followed by a median-bandwidth RBF kernel.

Each quantum kernel is passed to scikit-learn's `SVR(kernel="precomputed")`. Results are evaluated in metres using MAE, RMSE, and R-squared.

> Quantum kernel matrices scale quadratically with the number of samples. The default configuration uses deterministic depth-balanced subsets so the notebook is practical on a local simulator. Set the sample limits to `None` only when sufficient compute and memory are available.

In [ ]:
%pip install numpy pandas matplotlib scikit-learn qiskit qiskit-aer qiskit-machine-learning pennylane pennylane-lightning

import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import StratifiedShuffleSplit

from qiskit.circuit.library import z_feature_map, zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel

import pennylane as qml

warnings.filterwarnings("ignore", category=DeprecationWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Existing Encoded Split

This follows `VQR_Method1.ipynb` and `VQR_Method2.ipynb`: the encoded CSVs are loaded directly so every quantum model starts from the same split and `[0, π]` feature angles.

In [ ]:
train_df = pd.read_csv("train_angle_encoded.csv")
test_df = pd.read_csv("test_angle_encoded.csv")

angle_features = [
    "blue_angle",
    "green_angle",
    "log_blue_angle",
    "log_green_angle",
    "bg_ratio_angle",
    "stumpf_angle"
]
target = "depth"

X_train_full = train_df[angle_features].to_numpy(dtype=float)
y_train_full = train_df[target].to_numpy(dtype=float)
X_test_full = test_df[angle_features].to_numpy(dtype=float)
y_test_full = test_df[target].to_numpy(dtype=float)

print("Full training shape:", X_train_full.shape)
print("Full testing shape:", X_test_full.shape)
print("Train depth range:", (y_train_full.min(), y_train_full.max()))
print("Test depth range:", (y_test_full.min(), y_test_full.max()))

## 2. Runtime Configuration

Fidelity kernels require an (n \times n) training matrix. The defaults below retain observations across the 0–10 m, 10–20 m, and 20–30 m depth ranges while keeping simulator runtime manageable.

For a full experiment, set `MAX_TRAIN_SAMPLES = None` and `MAX_TEST_SAMPLES = None`.

In [ ]:
MAX_TRAIN_SAMPLES = 400
MAX_TEST_SAMPLES = 200

SVR_C = 10.0
SVR_EPSILON = 0.1

M1_REPS = 2
M2_REPS = 2
M3_LAYERS = 2

def depth_balanced_subset(X, y, max_samples, random_state):
    """Return a deterministic subset balanced across three depth intervals."""
    if max_samples is None or len(y) <= max_samples:
        return X.copy(), y.copy()

    depth_bins = pd.cut(
        y,
        bins=[-np.inf, 10.0, 20.0, np.inf],
        labels=False,
        include_lowest=True
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=max_samples,
        random_state=random_state
    )
    selected, _ = next(splitter.split(X, depth_bins))
    return X[selected], y[selected]

X_train, y_train = depth_balanced_subset(
    X_train_full, y_train_full, MAX_TRAIN_SAMPLES, RANDOM_STATE
)
X_test, y_test = depth_balanced_subset(
    X_test_full, y_test_full, MAX_TEST_SAMPLES, RANDOM_STATE + 1
)

target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

print("QSVR training shape:", X_train.shape)
print("QSVR testing shape:", X_test.shape)
print("Estimated train-kernel entries:", len(X_train) ** 2)

## 3. Shared QSVR Evaluation

All three methods produce a precomputed train kernel and a test-to-train kernel. The helper checks matrix shape and numerical validity, fits the same SVR configuration, converts predictions back to metres, and records timing plus regression metrics.

In [ ]:
results = []
predictions_by_method = {}
kernels_by_method = {}

def validate_kernel_matrices(K_train, K_test, method_name):
    K_train = np.asarray(K_train, dtype=float)
    K_test = np.asarray(K_test, dtype=float)

    expected_train = (len(X_train), len(X_train))
    expected_test = (len(X_test), len(X_train))

    if K_train.shape != expected_train:
        raise ValueError(
            f"{method_name}: expected train kernel {expected_train}, got {K_train.shape}"
        )
    if K_test.shape != expected_test:
        raise ValueError(
            f"{method_name}: expected test kernel {expected_test}, got {K_test.shape}"
        )
    if not np.isfinite(K_train).all() or not np.isfinite(K_test).all():
        raise ValueError(f"{method_name}: kernel contains NaN or infinite values")

    symmetry_error = np.max(np.abs(K_train - K_train.T))
    return K_train, K_test, symmetry_error

def fit_and_evaluate_qsvr(method_name, K_train, K_test, kernel_seconds):
    K_train, K_test, symmetry_error = validate_kernel_matrices(
        K_train, K_test, method_name
    )

    fit_start = time.perf_counter()
    model = SVR(
        kernel="precomputed",
        C=SVR_C,
        epsilon=SVR_EPSILON
    )
    model.fit(K_train, y_train_scaled)
    fit_seconds = time.perf_counter() - fit_start

    prediction_start = time.perf_counter()
    predicted_scaled = model.predict(K_test)
    prediction_seconds = time.perf_counter() - prediction_start

    predictions = target_scaler.inverse_transform(
        predicted_scaled.reshape(-1, 1)
    ).ravel()

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Method": method_name,
        "MAE (m)": mae,
        "RMSE (m)": rmse,
        "R2": r2,
        "Kernel time (s)": kernel_seconds,
        "SVR fit time (s)": fit_seconds,
        "Prediction time (s)": prediction_seconds,
        "Train symmetry error": symmetry_error
    })
    predictions_by_method[method_name] = predictions
    kernels_by_method[method_name] = (K_train, K_test)

    print("=" * 62)
    print(f"{method_name} RESULTS")
    print("=" * 62)
    print(f"MAE                 : {mae:.4f} m")
    print(f"RMSE                : {rmse:.4f} m")
    print(f"R2                  : {r2:.4f}")
    print(f"Kernel build time   : {kernel_seconds:.2f} s")
    print(f"SVR fit time        : {fit_seconds:.2f} s")
    print(f"Prediction time     : {prediction_seconds:.4f} s")
    print(f"Kernel symmetry err.: {symmetry_error:.3e}")

    return model, predictions

## 4. QSVR M1 — Local Fidelity Quantum Kernel

M1 uses independent single-qubit phase encoding through `ZFeatureMap`. It is the least entangled QSVR baseline and isolates the contribution of local quantum feature encoding.

In [ ]:
n_qubits = len(angle_features)

m1_feature_map = z_feature_map(
    feature_dimension=n_qubits,
    reps=M1_REPS
)
m1_quantum_kernel = FidelityQuantumKernel(
    feature_map=m1_feature_map
)

print("QSVR M1 feature map")
print(m1_feature_map.decompose().draw(output="text"))

In [ ]:
m1_start = time.perf_counter()

K_train_m1 = m1_quantum_kernel.evaluate(x_vec=X_train)
K_test_m1 = m1_quantum_kernel.evaluate(
    x_vec=X_test,
    y_vec=X_train
)

m1_kernel_seconds = time.perf_counter() - m1_start

m1_model, m1_predictions = fit_and_evaluate_qsvr(
    "QSVR M1 - Local Fidelity",
    K_train_m1,
    K_test_m1,
    m1_kernel_seconds
)

## 5. QSVR M2 — Entangled Fidelity Quantum Kernel

M2 uses `ZZFeatureMap` with linear entanglement. The two-qubit phase interactions encode feature relationships directly, making this kernel more expressive than M1 while increasing circuit depth.

In [ ]:
m2_feature_map = zz_feature_map(
    feature_dimension=n_qubits,
    reps=M2_REPS,
    entanglement="linear"
)
m2_quantum_kernel = FidelityQuantumKernel(
    feature_map=m2_feature_map
)

print("QSVR M2 feature map")
print(m2_feature_map.decompose().draw(output="text"))

In [ ]:
m2_start = time.perf_counter()

K_train_m2 = m2_quantum_kernel.evaluate(x_vec=X_train)
K_test_m2 = m2_quantum_kernel.evaluate(
    x_vec=X_test,
    y_vec=X_train
)

m2_kernel_seconds = time.perf_counter() - m2_start

m2_model, m2_predictions = fit_and_evaluate_qsvr(
    "QSVR M2 - Entangled Fidelity",
    K_train_m2,
    K_test_m2,
    m2_kernel_seconds
)

## 6. QSVR M3 — Projected Quantum Kernel

M3 follows a different execution pattern. A PennyLane circuit maps each encoded sample into local Pauli-X and Pauli-Z expectation values after circular entanglement. A classical RBF kernel is then evaluated in this quantum-projected feature space.

The RBF bandwidth is selected with the median-distance heuristic using only the projected training data.

In [ ]:
m3_device = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(m3_device)
def projected_quantum_features(encoded_sample):
    qml.AngleEmbedding(
        encoded_sample,
        wires=range(n_qubits),
        rotation="Y"
    )

    for _ in range(M3_LAYERS):
        for wire in range(n_qubits):
            qml.Hadamard(wires=wire)
            qml.RZ(encoded_sample[wire], wires=wire)

        for wire in range(n_qubits):
            qml.CNOT(wires=[wire, (wire + 1) % n_qubits])

    return (
        [qml.expval(qml.PauliX(wire)) for wire in range(n_qubits)]
        + [qml.expval(qml.PauliZ(wire)) for wire in range(n_qubits)]
    )

sample_signature = np.asarray(
    projected_quantum_features(X_train[0]),
    dtype=float
)
print("QSVR M3 projected feature dimension:", sample_signature.shape[0])
print(qml.draw(projected_quantum_features)(X_train[0]))

In [ ]:
m3_start = time.perf_counter()

projected_train = np.asarray([
    projected_quantum_features(sample) for sample in X_train
], dtype=float)
projected_test = np.asarray([
    projected_quantum_features(sample) for sample in X_test
], dtype=float)

projection_scaler = StandardScaler()
projected_train_scaled = projection_scaler.fit_transform(projected_train)
projected_test_scaled = projection_scaler.transform(projected_test)

train_squared_distances = pairwise_distances(
    projected_train_scaled,
    metric="sqeuclidean"
)
positive_distances = train_squared_distances[train_squared_distances > 0]
median_squared_distance = np.median(positive_distances)
m3_gamma = 1.0 / max(median_squared_distance, 1e-12)

K_train_m3 = rbf_kernel(
    projected_train_scaled,
    projected_train_scaled,
    gamma=m3_gamma
)
K_test_m3 = rbf_kernel(
    projected_test_scaled,
    projected_train_scaled,
    gamma=m3_gamma
)

m3_kernel_seconds = time.perf_counter() - m3_start

print(f"Median squared distance: {median_squared_distance:.6f}")
print(f"Selected RBF gamma: {m3_gamma:.6f}")

m3_model, m3_predictions = fit_and_evaluate_qsvr(
    "QSVR M3 - Projected Kernel",
    K_train_m3,
    K_test_m3,
    m3_kernel_seconds
)

## 7. Method Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

display(
    results_df.style
    .format({
        "MAE (m)": "{:.4f}",
        "RMSE (m)": "{:.4f}",
        "R2": "{:.4f}",
        "Kernel time (s)": "{:.2f}",
        "SVR fit time (s)": "{:.2f}",
        "Prediction time (s)": "{:.4f}",
        "Train symmetry error": "{:.3e}"
    })
    .highlight_min(subset=["MAE (m)", "RMSE (m)"], color="#c7f3e8")
    .highlight_max(subset=["R2"], color="#c7f3e8")
)

best_method = results_df.iloc[0]["Method"]
print(f"Best QSVR method by R2: {best_method}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

metric_config = [
    ("MAE (m)", "MAE (m)", False),
    ("RMSE (m)", "RMSE (m)", False),
    ("R2", "R-squared", True)
]

colors = ["#176B87", "#64CCC5", "#D97745"]

for axis, (column, title, higher_is_better) in zip(axes, metric_config):
    axis.bar(results_df["Method"], results_df[column], color=colors)
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=25)
    axis.grid(axis="y", alpha=0.25)

    best_index = (
        results_df[column].idxmax()
        if higher_is_better
        else results_df[column].idxmin()
    )
    axis.patches[best_index].set_edgecolor("black")
    axis.patches[best_index].set_linewidth(2)

plt.suptitle("QSVR Methods M1, M2, and M3")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharex=True, sharey=True)

for axis, (method_name, predictions), color in zip(
    axes,
    predictions_by_method.items(),
    colors
):
    method_r2 = r2_score(y_test, predictions)
    axis.scatter(y_test, predictions, alpha=0.5, color=color)
    axis.plot([0, 30], [0, 30], "k--", label="Ideal prediction")
    axis.set_title(f"{method_name}\nR2 = {method_r2:.3f}")
    axis.set_xlabel("Actual depth (m)")
    axis.set_xlim(0, 30)
    axis.set_ylim(0, 30)
    axis.grid(alpha=0.25)

axes[0].set_ylabel("Predicted depth (m)")
axes[-1].legend()
plt.tight_layout()
plt.show()

## 8. Interpretation and Reproducibility Notes

- **M1** tests a local, non-entangled fidelity kernel.
- **M2** adds pairwise feature interactions using an entangled fidelity kernel.
- **M3** measures quantum-projected observables and applies a median-bandwidth RBF kernel.
- The same SVR hyperparameters and target standardization are used for all three methods.
- The deterministic subsets are intended for simulator feasibility, not for claiming full-dataset benchmark parity.
- For the final scientific comparison, run all models on matching observations and replace the random split with spatial validation.
- Kernel construction time should be reported alongside predictive accuracy because it is the dominant QSVR cost.